# Lab 03.5 – Integrated XADC System Monitor

**Student:** ____________________   **Group:** __________   **Date:** __________

This individual assignment combines XADC temperature and voltage measurements with GPIO, periodic timing and application-state handling.

> **Important:** Hardware/XADC initialization and helper functions are provided. The monitoring application logic must be designed and implemented by the student.

No external sensor or additional hardware is required.


## Objectives
- combine Linux IIO/XADC measurements with PYNQ GPIO;
- schedule periodic measurements without blocking button handling;
- implement temperature and voltage alarm conditions;
- maintain minimum and maximum measurement statistics;
- use LEDs as a compact system-status display.


## 1. Hardware and XADC initialization

Run this cell before the assignment. It loads the same `base.bit` overlay used in the preceding laboratories and locates the Linux XADC IIO device.


In [ ]:
from pynq import Overlay
from pathlib import Path
import re
import time

ol = Overlay("base.bit")

leds = ol.leds_gpio.channel1
leds.setdirection("out")
leds.setlength(4)

buttons = ol.btns_gpio.channel1
buttons.setdirection("in")
buttons.setlength(4)

switches = ol.switches_gpio.channel1
switches.setdirection("in")
switches.setlength(4)

leds.write(0, 0b1111)

iio_root = Path("/sys/bus/iio/devices")
xadc_path = None

for dev in sorted(iio_root.glob("iio:device*")):
    name_file = dev / "name"
    if name_file.exists() and name_file.read_text().strip() == "xadc":
        xadc_path = dev
        break

if xadc_path is None:
    raise RuntimeError("XADC IIO device not found.")

print("GPIO and XADC initialized.")


## 2. Measurement helper functions

These functions are provided. The assignment starts after this section.


In [ ]:
def read_temperature_celsius():
    raw = float((xadc_path / "in_temp0_raw").read_text().strip())
    offset = float((xadc_path / "in_temp0_offset").read_text().strip())
    scale = float((xadc_path / "in_temp0_scale").read_text().strip())
    return (raw + offset) * scale / 1000.0

def parameter_file(raw_file, parameter):
    base = raw_file.name[:-4]
    candidates = [xadc_path / f"{base}_{parameter}"]
    m = re.match(r"^(in_voltage\d+)(?:_[A-Za-z0-9]+)?$", base)
    if m:
        candidates.append(xadc_path / f"{m.group(1)}_{parameter}")
    candidates.append(xadc_path / f"in_voltage_{parameter}")
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def read_voltage(raw_file):
    raw = float(raw_file.read_text().strip())
    scale_file = parameter_file(raw_file, "scale")
    if scale_file is None:
        raise RuntimeError(f"No scale found for {raw_file.name}")
    scale = float(scale_file.read_text().strip())
    offset_file = parameter_file(raw_file, "offset")
    offset = float(offset_file.read_text().strip()) if offset_file else 0.0
    return (raw + offset) * scale / 1000.0

voltage_raw_files = sorted(xadc_path.glob("in_voltage*_raw"))
vccint_file = next((f for f in voltage_raw_files if "vccint" in f.name), None)

if vccint_file is None:
    raise RuntimeError("VCCINT channel not found.")

print(f"Temperature = {read_temperature_celsius():.2f} °C")
print(f"VCCINT      = {read_voltage(vccint_file):.4f} V")


# Task 5 – Interactive XADC System Monitor

## 3. Required behavior

At program startup:

1. Read the current temperature and VCCINT once and store them as `baseline_temperature` and `baseline_vccint`.
2. Start with a temperature warning margin of **5 °C** above the baseline.
3. The critical temperature threshold is always **5 °C above the warning threshold**.
4. The VCCINT alarm condition is a deviation of more than **5%** from the startup baseline voltage.

### Switch configuration

Use `SW1:SW0` to select the sampling interval continuously while the program is running:

| SW1:SW0 | Sampling interval |
|---:|---:|
| 00 | 0.25 s |
| 01 | 0.50 s |
| 10 | 1.00 s |
| 11 | 2.00 s |

`SW2` and `SW3` are unused in the required task.

### Button controls

- **BTN0** – increase the warning margin by 1 °C;
- **BTN1** – decrease the warning margin by 1 °C, but never below 1 °C;
- **BTN2** – reset the recorded minimum/maximum temperature and VCCINT values to the current measurements;
- **BTN3** – stop the program and switch off all LEDs.

A held button must generate only one command. Use new-press detection as in the previous laboratories.


## 4. LED status encoding

The four LEDs have independent meanings:

| LED | Meaning |
|---|---|
| LD0 | Heartbeat: toggle after every accepted XADC sample |
| LD1 | Temperature warning: ON when `T >= warning_threshold` |
| LD2 | Temperature critical: ON when `T >= critical_threshold` |
| LD3 | VCCINT alarm: ON when VCCINT differs by more than ±5% from its startup baseline |

LD1 and LD2 may therefore both be ON in the critical state.


## 5. Timing requirements

The application must remain responsive between XADC samples.

- Do **not** implement the sampling interval using a long `time.sleep(interval)` in the main loop.
- Use `time.monotonic()` and a `next_sample` or `last_sample` variable.
- A short polling delay such as `time.sleep(0.01)` is permitted.
- If the switch-selected interval changes, the new interval must take effect without restarting the program.
- Print one status line for each accepted XADC sample and one line when a button changes the warning margin or resets statistics.


## 6. Design stage

Complete the table before coding.

| Design question | Your answer |
|---|---|
| Variables that define the monitor state | |
| Expression to decode `SW1:SW0` as an index 0…3 | |
| Method used to detect a new button press | |
| Expression for the warning threshold | |
| Expression for the critical threshold | |
| Lower and upper VCCINT alarm limits | |
| Method used to schedule XADC samples without blocking the main loop | |
| Expression used to construct the 4-bit LED pattern | |


## 7. Student implementation

Complete the code below. Helper functions from Sections 1 and 2 must be reused; do not hard-code an assumed XADC raw-to-physical conversion.


In [ ]:
SAMPLE_INTERVALS = (0.25, 0.50, 1.00, 2.00)
VOLTAGE_TOLERANCE = 0.05

baseline_temperature = read_temperature_celsius()
baseline_vccint = read_voltage(vccint_file)

warning_margin = 5.0
heartbeat = False
previous_buttons = buttons.read() & 0xF

current_temperature = baseline_temperature
current_vccint = baseline_vccint

min_temperature = max_temperature = current_temperature
min_vccint = max_vccint = current_vccint

next_sample = time.monotonic()

print(f"Baseline temperature: {baseline_temperature:.2f} °C")
print(f"Baseline VCCINT:      {baseline_vccint:.4f} V")

raise NotImplementedError("Complete the TODO sections, then remove this line.")

try:
    while True:
        current_buttons = buttons.read() & 0xF

        # TODO 1: detect new button presses.
        pressed = 0

        # TODO 2: BTN3 stops the application.
        # TODO 3: BTN0/BTN1 modify warning_margin.
        # TODO 4: BTN2 resets min/max values to the current measurements.

        switch_value = switches.read() & 0xF

        # TODO 5: decode SW1:SW0 and select the sampling interval.
        sample_interval = 1.0

        now = time.monotonic()

        if now >= next_sample:
            # TODO 6: read temperature and VCCINT.
            # TODO 7: update min/max statistics.
            # TODO 8: calculate warning, critical and voltage-alarm conditions.
            # TODO 9: toggle heartbeat and construct the complete 4-bit LED pattern.
            # TODO 10: write the pattern and print one status line.
            # TODO 11: schedule the next sample from the current interval.
            pass

        previous_buttons = current_buttons
        time.sleep(0.01)

finally:
    leds.write(0, 0b1111)
    print("Monitor stopped; all LEDs are OFF.")


## 8. Acceptance checklist

- [ ] Startup temperature and VCCINT are read successfully.
- [ ] `SW1:SW0` selects all four sampling intervals during execution.
- [ ] Holding BTN0 or BTN1 changes the warning margin only once.
- [ ] The warning margin never falls below 1 °C.
- [ ] BTN2 resets all four min/max statistics to the current values.
- [ ] LD0 toggles only when an XADC sample is accepted.
- [ ] LD1 follows the warning condition.
- [ ] LD2 follows the critical condition.
- [ ] LD3 follows the ±5% VCCINT condition.
- [ ] The application remains responsive at the 2 s sampling interval.
- [ ] BTN3 stops the loop and switches off all LEDs.
- [ ] A Jupyter kernel interrupt also leaves the LEDs OFF because of the `finally` block.

Record at least one implementation error you encountered and how you diagnosed it.

**Observed behavior / corrections:**  
_Write your notes here._


# Task 5+ – CSV Event Logger (optional)

Extend the working monitor so that each accepted sample is also written to a CSV file with these fields:

`elapsed_s, temperature_C, vccint_V, warning, critical, voltage_alarm, sample_interval_s`

Additional requirements:

- Open the file once before entering the main loop and close it reliably on exit.
- Write one row per accepted XADC sample, not one row per polling-loop iteration.
- Add an `event` field that records `BTN0`, `BTN1`, `RESET_STATS`, or `STOP` when appropriate; otherwise leave it empty.
- After the run, load the CSV file and plot temperature versus elapsed time.


## Questions
1. Why is the startup VCCINT measurement used as the reference instead of hard-coding a voltage value in the application?
2. Why are LD1 and LD2 independent bits instead of mutually exclusive states?
3. Why is a long `time.sleep(sample_interval)` inappropriate in the main loop?
4. What is the difference between the polling-loop rate and the XADC sampling rate?
5. Why must the XADC conversion use the IIO `scale` and `offset` information?
6. What information do the minimum and maximum values provide that the current measurement does not?
7. Which parts of this monitor would benefit from being implemented in programmable logic in a higher-performance embedded system?
